# Set up library

In [1]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint,  EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
import h5py
from sklearn.utils.class_weight import compute_class_weight
import os





# Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Data Module

In [3]:
from os.path import isfile
class DataModule:

    def __init__(self):
        self.data_set_dic = {}

    def __extract_dft(self, window):
        eps = 1e-10
        spectrum = np.abs(np.fft.rfft(window))
        spectrum = spectrum / np.max(spectrum + eps)
        return spectrum

    def __suppress_low_amplitudes(self, spec, threshold_db=50):
        I_threshold = 10 ** (-threshold_db / 20)
        spec[spec < I_threshold] *= 0.2
        return spec
    def __get_spectrum_of_window(self, window):

        spectrum = self.__extract_dft(window)

        #preprocessing
        standard_spectrum = self.__suppress_low_amplitudes(spectrum)

        return standard_spectrum

    def process_one_file_audio(
        self,
        audio_path,
        csv_path,
        sample_rate=48000,
        window_size=0.01
    ):
        #load audio
        audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

        samples_per_window = int(sample_rate * window_size)

        # Load CSV
        df = pd.read_csv(csv_path)
        X = []
        y = []

        # preparing data
        for _, row in df.iterrows():
            start_sample = int(row["start_time"] * sample_rate)
            end_sample = start_sample + samples_per_window

            if end_sample > len(audio):
                continue

            window = audio[start_sample:end_sample]
            spectrum = self.__get_spectrum_of_window(window)

            X.append(spectrum)
            y.append(row["label"])

        # reducing silence sample
        silence_label = "silence"
        flag = True
        start_index = 0
        end_index = len(y) - 1
        max_silence_label = 3
        X_reduced = []
        y_reduced = []
        while flag:

            if y[start_index] == silence_label:
                start_index += 1

            if y[end_index] == silence_label:
                end_index -= 1

            if y[start_index] != silence_label and y[end_index] != silence_label:
                X_reduced = X[start_index:end_index + 1]
                y_reduced = y[start_index:end_index + 1]


                num_start_silence = start_index - 0
                num_end_silence = len(y) - 1 - end_index

                if num_start_silence > max_silence_label:
                    num_start_silence = max_silence_label

                if num_end_silence > max_silence_label:
                    num_end_silence = max_silence_label

                for i in np.arange(num_start_silence):
                    X_reduced.append(X[i])
                    y_reduced.append(y[i])

                for i in np.arange(num_end_silence):
                    end_index = len(y) - 1 - i
                    X_reduced.append(X[end_index])
                    y_reduced.append(y[end_index])

                flag = False



        return np.array(X_reduced), np.array(y_reduced)

    def build_dataset(self):

        # read data from drive
        audio_data_set_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio"
        audio_folder_name = "audio"
        labels_folder_name = "labels"

        if os.path.exists(audio_data_set_path):
          for root, dirs, files in os.walk(audio_data_set_path):
            for file in files:
              if file.endswith(".wav"):
                audio_file_path = os.path.join(root, file)
                labels_file_name = file.replace(".wav", ".csv")
                labels_file_root = root.replace(audio_folder_name, labels_folder_name)
                #check if specified audio has corresponded labels file
                labels_path = os.path.join(labels_file_root, labels_file_name)
                # print("check audio path and labels path: ", audio_file_path, labels_path)
                if os.path.exists(labels_path):
                    if os.path.exists(labels_path):
                      self.data_set_dic[audio_file_path] = labels_path
                    else:
                      print(labels_path, "is not a file")
                else:
                  print("Cant find file at", labels_path)

        else:
          print("Cant find folder at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio")

        X_all = []
        y_all = []

        for audio_file_path, labels_file_path in self.data_set_dic.items():
          #  print ("check audio path and corresponding label path: ", audio_file_path, labels_file_path)
           X, y = self.process_one_file_audio(audio_file_path, labels_file_path)
           X_all.append(X)
           y_all.append(y)

        X_all = np.vstack(X_all)
        y_all = np.hstack(y_all)

        return X_all, y_all

#Main

## Set up HDF5 File and labeling Data

In [4]:
HDF5_file_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5"
X = None
y = None

if os.path.exists(HDF5_file_path):
    data_set_path = HDF5_file_path + "/Qeej_Hmong_Features.h5"
    if os.path.exists(data_set_path):
      print("Retrieving data from drive with path:", data_set_path)
      with h5py.File(data_set_path, "r") as f:
        X = f["features"][:]
        y = f["labels"][:]

    else:
        print("set up data")
        dataModule = DataModule()
        X, y = dataModule.build_dataset()
        print("check len X:", len(X))
        print("check len y:", len(y))
        with h5py.File(data_set_path, "w") as f:
          f.create_dataset("features", data=X)
          f.create_dataset("labels", data=y.tolist())

Retrieving data from drive with path: /content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5/Qeej_Hmong_Features.h5


##label encoding

In [5]:
le = LabelEncoder()
y_int = le.fit_transform(y)
y_cat = to_categorical(y_int)

class_names = le.classes_

## Set up Training/Validation/Testing Data

In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.3, stratify=y_int)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.1)

y_train_int = np.argmax(y_train, axis=1)
print("check y train int:", y_train_int)


check y train int: [42 27  3 ... 10 13 21]


## Set up Class Weights

In [7]:
unique, counts = np.unique(y_train_int, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count} samples")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_int),
    y=y_train_int
)

class_weight_dict = dict(enumerate(class_weights))

print("check class weight dict:", class_weight_dict)


Class 0: 2946 samples
Class 1: 2616 samples
Class 2: 2365 samples
Class 3: 2208 samples
Class 4: 2301 samples
Class 5: 2036 samples
Class 6: 1965 samples
Class 7: 2164 samples
Class 8: 1954 samples
Class 9: 2180 samples
Class 10: 2439 samples
Class 11: 2300 samples
Class 12: 2045 samples
Class 13: 2252 samples
Class 14: 2193 samples
Class 15: 2097 samples
Class 16: 2224 samples
Class 17: 2612 samples
Class 18: 2519 samples
Class 19: 2213 samples
Class 20: 2076 samples
Class 21: 2263 samples
Class 22: 2164 samples
Class 23: 2106 samples
Class 24: 2092 samples
Class 25: 2660 samples
Class 26: 2502 samples
Class 27: 2244 samples
Class 28: 2423 samples
Class 29: 2729 samples
Class 30: 2166 samples
Class 31: 2531 samples
Class 32: 1783 samples
Class 33: 2802 samples
Class 34: 2439 samples
Class 35: 2156 samples
Class 36: 2208 samples
Class 37: 2123 samples
Class 38: 2122 samples
Class 39: 2015 samples
Class 40: 2215 samples
Class 41: 2180 samples
Class 42: 2514 samples
Class 43: 2148 sample

## Set up early stopping

In [8]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True
)

## Set up check points

In [9]:
CHECKPOINT_DIR = "/content/drive/MyDrive/Qeej_Hmong_Model_Parameters/CheckPoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "model_latest.keras")

checkpoint_callback = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    save_weights_only=False,
    save_best_only=False,
    save_freq='epoch'
)

## set up model

In [10]:

batch_size = 128
epoch = 500


if os.path.exists(CHECKPOINT_PATH):
    print("Loading existing model...")
    model = load_model(CHECKPOINT_PATH)
    initial_epoch = int(model.optimizer.iterations.numpy() / (len(X_train) // batch_size))
else:
    print("Creating new model...")
    model = Sequential([
        Conv1D(32, kernel_size=5, activation='relu',
              input_shape=(X.shape[1], 1)),
        MaxPooling1D(2),
        Dropout(0.25),

        Conv1D(64, kernel_size=5, activation='relu'),
        MaxPooling1D(2),
        Dropout(0.25),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),

        Dense(y_cat.shape[1], activation='softmax')
    ])
    # model = Sequential([
    #     Conv1D(32, 5, activation='relu', input_shape=(X.shape[1],1)),
    #     BatchNormalization(),
    #     MaxPooling1D(2),
    #     Dropout(0.25),

    #     Conv1D(64, 5, activation='relu'),
    #     BatchNormalization(),
    #     MaxPooling1D(2),
    #     Dropout(0.25),

    #     Conv1D(128, 5, activation='relu'),
    #     BatchNormalization(),
    #     MaxPooling1D(2),
    #     Dropout(0.25),

    #     Flatten(),

    #     Dense(128, activation='relu'),
    #     Dropout(0.5),

    #     Dense(y_cat.shape[1], activation='softmax')
    # ])

    optimizer = Adam(learning_rate=0.001)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    initial_epoch = 0



Creating new model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## training model

In [ ]:

model.fit(
    X_train[..., np.newaxis],
    y_train,
    validation_data=(X_val[..., np.newaxis], y_val),
    class_weight=class_weight_dict,
    epochs=epoch,
    batch_size=batch_size,
    callbacks=[checkpoint_callback, early_stop],
    initial_epoch=initial_epoch
)



Epoch 1/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - accuracy: 0.0667 - loss: 3.9057 - val_accuracy: 0.2622 - val_loss: 2.9444
Epoch 2/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1988 - loss: 3.1425 - val_accuracy: 0.3542 - val_loss: 2.5899
Epoch 3/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.2497 - loss: 2.9031 - val_accuracy: 0.4090 - val_loss: 2.3171
Epoch 4/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.2784 - loss: 2.7706 - val_accuracy: 0.4365 - val_loss: 2.2389
Epoch 5/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.2973 - loss: 2.6859 - val_accuracy: 0.4663 - val_loss: 2.1051
Epoch 6/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.3140 - loss: 2.6175 - val_accuracy: 0.4844 - val_loss: 2.0402
Epoch 7/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.3221 - loss: 2.5673 - val_accuracy: 0.4911 - val_loss: 2.0024
Epoch 8/500
1180/1180 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - accuracy: 0.3312 - loss:

## Save weights

## assess model

In [ ]:
#assess model
y_pred = model.predict(X_test[..., np.newaxis])
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print("check y pred:", y_pred)
print("check y true:", y_true)

### F1 score

In [ ]:
#F1 score, ...
decoded_class_names = [name.decode('utf-8') for name in class_names]
print(classification_report(
    y_true, y_pred,
    target_names=decoded_class_names
))

### confuse matrix


In [ ]:

#confuse matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(30, 16))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=decoded_class_names
)
disp.plot(ax=ax)

plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix (10 ms windows)")
plt.show()